# Collect Library Documentation for Vector Database

This notebook scrapes **migration guides**, **changelogs / what's-new pages**, and **key API references** for every dependency listed in `requirements_eval.txt`.  
The extracted text is cleaned, chunked, enriched with metadata, and saved as a **JSON** file ready for embedding and ingestion into a vector database (used by the RAG pipelines).

### Pipeline overview
1. Parse `requirements_eval.txt` to get the list of libraries  
2. Map each library to its official documentation / migration-guide URLs  
3. Fetch & extract clean text from each page  
4. Chunk the text into passages suitable for embedding  
5. Attach metadata (library name, version, doc category, source URL)  
6. Save everything as a structured JSON file

## 1 — Install & Import Dependencies

In [1]:
# Install scraping utilities (if not already present)
#!pip install requests beautifulsoup4 lxml tqdm truststore -q

In [2]:
import os
import re
import json
import time
import hashlib
import textwrap
from pathlib import Path
from datetime import datetime, timezone
from typing import Optional

import requests
from bs4 import BeautifulSoup, Comment
from tqdm.auto import tqdm

print("All imports successful.")

All imports successful.


c:\Users\hbahmanyar\MentorApp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2 — Parse `requirements_eval.txt`

In [3]:
REQUIREMENTS_PATH = Path("../requirements_eval.txt")

def parse_requirements(path: Path) -> list[str]:
    """Return a deduplicated list of package names from a pip requirements file."""
    libs = []
    with open(path) as f:
        for raw in f:
            line = raw.split("#")[0].split(";")[0].strip()  # strip comments & markers
            if not line:
                continue
            # handle version specifiers: pkg>=1.0, pkg==2.0, pkg[extra]
            name = re.split(r"[>=<!\[]", line)[0].strip()
            if name:
                libs.append(name.lower())
    return list(dict.fromkeys(libs))  # deduplicate, preserve order

libraries = parse_requirements(REQUIREMENTS_PATH)
print(f"Found {len(libraries)} libraries:")
for i, lib in enumerate(libraries, 1):
    print(f"  {i:2d}. {lib}")

Found 30 libraries:
   1. ï»¿
   2. numpy
   3. pandas
   4. scipy
   5. matplotlib
   6. seaborn
   7. scikit-learn
   8. pyarrow
   9. openpyxl
  10. pillow
  11. transformers
  12. datasets
  13. accelerate
  14. sentencepiece
  15. langchain
  16. langchain-openai
  17. python-dotenv
  18. requests
  19. httpx
  20. tqdm
  21. truststore
  22. torch
  23. torchvision
  24. torchaudio
  25. tensorflow
  26. opencv-python
  27. scikit-image
  28. xgboost
  29. lightgbm
  30. catboost


## 3 — Load Documentation URL Registry

The URL registry lives in an external module — [`doc_urls_registry.py`](doc_urls_registry.py) — to keep this notebook lean.  
Each library maps to a list of `(category, version, url)` tuples.  
Categories include: `migration`, `whatsnew`, `changelog`, `guide`, `api`.  
The `version` field (e.g. `"2.4.0"`, `"1.8"`) is stored in every chunk's metadata so the retriever can filter by version at query time.

In [4]:
# Import the URL registry from the external module
from doc_urls_registry import DOC_URLS

# Quick summary
total_urls = sum(len(v) for v in DOC_URLS.values())
print(f"Registered {total_urls} documentation URLs across {len(DOC_URLS)} libraries.")

# Show any libraries in requirements that have NO doc URLs mapped
unmapped = [lib for lib in libraries if lib not in DOC_URLS]
if unmapped:
    print(f"\n⚠  Libraries without doc URLs (skipped): {unmapped}")
else:
    print("All libraries are mapped to documentation URLs.")

Registered 39 documentation URLs across 28 libraries.

⚠  Libraries without doc URLs (skipped): ['ï»¿', 'truststore']


## 4 — Text Extraction Utilities

In [5]:
# ── Use OS-level certificate store for SSL ──────────────────────────────────
import truststore
truststore.inject_into_ssl()
print("✓ Injected OS certificate store (truststore) for SSL verification.")

# ── HTTP session with retry & polite headers ────────────────────────────────
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    # NOTE: Only request gzip/deflate — requests handles these natively.
    # Brotli (br) requires the 'brotli' package which may not be installed.
    "Accept-Encoding": "gzip, deflate",
    "Connection": "keep-alive",
})

# Retry adapter with robust settings
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

retry_strategy = Retry(
    total=5,                          # retries on failure
    backoff_factor=2,                 # exponential backoff: 2, 4, 8, 16, 32 sec
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET", "HEAD"],
)
adapter = HTTPAdapter(
    max_retries=retry_strategy,
    pool_connections=10,
    pool_maxsize=10,
)
SESSION.mount("https://", adapter)
SESSION.mount("http://", adapter)

POLITE_DELAY = 2.0  # seconds between requests
REQUEST_TIMEOUT = (10, 60)  # (connect timeout, read timeout)


def fetch_page(url: str, timeout: tuple = REQUEST_TIMEOUT) -> Optional[str]:
    """Fetch a page and return its HTML text, or None on failure."""
    try:
        resp = SESSION.get(url, timeout=timeout)
        resp.raise_for_status()
        # Ensure proper encoding detection
        resp.encoding = resp.apparent_encoding or "utf-8"
        return resp.text
    except requests.exceptions.SSLError as exc:
        print(f"    ✗ SSL error: {exc}")
        return None
    except requests.exceptions.ConnectionError as exc:
        print(f"    ✗ Connection error: {exc}")
        return None
    except requests.exceptions.Timeout as exc:
        print(f"    ✗ Timeout: {exc}")
        return None
    except requests.RequestException as exc:
        print(f"    ✗ Request failed: {exc}")
        return None


print("HTTP session configured with retry strategy and increased timeouts.")

✓ Injected OS certificate store (truststore) for SSL verification.
HTTP session configured with retry strategy and increased timeouts.


In [6]:
# ── HTML → clean text ───────────────────────────────────────────────────────

# Tags that carry no useful documentary content
STRIP_TAGS = {
    "script", "style", "nav", "footer", "header",
    "aside", "form", "button", "svg", "img", "video",
    "noscript", "iframe",
}

# Content-bearing block elements we extract text from
BLOCK_TAGS = {"p", "li", "dd", "dt", "td", "th", "blockquote"}


def _is_inside_processed(el, processed_ids: set) -> bool:
    """Return True if any ancestor of *el* has already been processed."""
    for ancestor in el.parents:
        if id(ancestor) in processed_ids:
            return True
    return False


def _dedup_lines(lines: list[str]) -> list[str]:
    """
    Remove duplicate / noise lines that appear in scraped HTML:
      1. Consecutive identical lines (from nested elements)
      2. Lines that are ONLY backtick-wrapped tokens already present
         in the immediately preceding line (orphan <code> artefacts)
    """
    cleaned: list[str] = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            cleaned.append(line)
            continue

        # Skip consecutive exact duplicates
        if cleaned and stripped == cleaned[-1].strip():
            continue

        # Skip lines made entirely of backtick tokens (`foo` `bar` ...)
        # if every token already appears in the previous line
        backtick_tokens = re.findall(r"`([^`]+)`", stripped)
        non_backtick = re.sub(r"`[^`]*`", "", stripped).strip()
        if backtick_tokens and not non_backtick and cleaned:
            prev = cleaned[-1]
            if all(tok in prev for tok in backtick_tokens):
                continue

        cleaned.append(line)
    return cleaned


def extract_main_text(html: str, url: str = "") -> str:
    """
    Extract the main textual content from an HTML page.
    Tracks processed elements to prevent parent-child duplicate extraction.
    Returns cleaned plain text with headings and code blocks preserved.
    """
    soup = BeautifulSoup(html, "lxml")

    # Remove unwanted tags
    for tag in soup.find_all(STRIP_TAGS):
        tag.decompose()
    # Remove HTML comments
    for comment in soup.find_all(string=lambda t: isinstance(t, Comment)):
        comment.extract()

    # Try to locate the main content area
    main = (
        soup.find("main")
        or soup.find(attrs={"role": "main"})
        or soup.find("article")
        or soup.find("div", class_=re.compile(r"(content|document|body|article)", re.I))
        or soup.body
        or soup
    )

    # ── Walk the tree, tracking processed elements to avoid duplication ──
    processed = set()          # ids of elements whose subtrees are consumed
    lines: list[str] = []

    for el in main.descendants:
        if el.name is None:    # skip NavigableString nodes
            continue
        # If this element sits inside an already-processed subtree → skip
        if _is_inside_processed(el, processed):
            continue

        # ── Headings ────────────────────────────────────────────────────
        if el.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
            heading_text = el.get_text(separator=" ", strip=True)
            if heading_text:
                level = int(el.name[1])
                lines.append(f"\n{'#' * level} {heading_text}\n")
                processed.add(id(el))

        # ── Pre-formatted code blocks ──────────────────────────────────
        elif el.name == "pre":
            code = el.get_text()
            lines.append(f"\n```\n{code}\n```\n")
            processed.add(id(el))

        # ── Content-bearing block elements ─────────────────────────────
        elif el.name in BLOCK_TAGS:
            text = el.get_text(separator=" ", strip=True)
            if text:
                lines.append(text)
                processed.add(id(el))

        # Standalone <code> outside the above blocks is skipped entirely —
        # its text is already captured by the parent's get_text().

    raw = "\n".join(lines)

    # Post-process: remove duplicate / noise lines
    raw_lines = raw.split("\n")
    raw_lines = _dedup_lines(raw_lines)
    raw = "\n".join(raw_lines)

    # Collapse excessive whitespace
    raw = re.sub(r"\n{3,}", "\n\n", raw)
    raw = re.sub(r"[ \t]+", " ", raw)
    return raw.strip()


# Quick smoke test
_sample = "<html><body><main><h1>Hello</h1><p>World</p></main></body></html>"
assert "Hello" in extract_main_text(_sample)

# Test dedup: nested li > p should not produce duplicates
_nested = """<html><body><main>
<ul><li><p>Same text here</p></li></ul>
</main></body></html>"""
_result = extract_main_text(_nested)
assert _result.count("Same text here") == 1, f"Dedup failed: {_result!r}"

print("Text extraction utility ready (with deduplication).")

Text extraction utility ready (with deduplication).


## 5 — Text Chunking

We split each document into overlapping chunks of ≈ `CHUNK_SIZE` characters, splitting on paragraph / heading boundaries when possible.  
Each chunk keeps enough context to be independently useful when retrieved by semantic search.

In [7]:
CHUNK_SIZE = 1500      # target characters per chunk
CHUNK_OVERLAP = 200    # overlap between consecutive chunks


def chunk_text(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP,
) -> list[str]:
    """
    Split *text* into chunks of roughly *chunk_size* characters,
    preferring to break at paragraph boundaries (double newlines).
    Consecutive chunks share *overlap* characters of context.
    """
    if len(text) <= chunk_size:
        return [text]

    # Split on double-newlines (paragraph / heading boundaries)
    paragraphs = re.split(r"\n{2,}", text)

    chunks: list[str] = []
    current = ""

    for para in paragraphs:
        candidate = (current + "\n\n" + para).strip() if current else para.strip()

        if len(candidate) > chunk_size and current:
            # Flush the current chunk
            chunks.append(current.strip())
            # Start next chunk with overlap from the end of current
            overlap_text = current[-overlap:] if overlap else ""
            current = (overlap_text + "\n\n" + para).strip()
        else:
            current = candidate

    if current.strip():
        chunks.append(current.strip())

    # Safety: break any remaining oversized chunks by hard character limit
    final: list[str] = []
    for ch in chunks:
        if len(ch) <= chunk_size * 1.3:  # allow 30 % grace
            final.append(ch)
        else:
            for i in range(0, len(ch), chunk_size - overlap):
                final.append(ch[i : i + chunk_size])

    return final


# Quick verification
_demo = "A" * 500 + "\n\n" + "B" * 500 + "\n\n" + "C" * 500 + "\n\n" + "D" * 500
_chunks = chunk_text(_demo, chunk_size=800, overlap=100)
print(f"Demo: {len(_demo)} chars → {len(_chunks)} chunks  "
      f"(sizes: {[len(c) for c in _chunks]})")
print("Chunking utility ready.")

Demo: 2006 chars → 4 chunks  (sizes: [500, 602, 602, 602])
Chunking utility ready.


## 6 — Scrape, Extract & Chunk All Documentation

In [8]:
def build_document(
    library: str,
    category: str,
    version: str,
    url: str,
    chunk_text_content: str,
    chunk_index: int,
    total_chunks: int,
) -> dict:
    """
    Build a single document dict suitable for vector-DB ingestion.
    The `version` field allows the retriever to filter by library version.
    """
    # Deterministic ID so re-runs don't create duplicates
    uid = hashlib.sha256(
        f"{library}::{url}::{chunk_index}".encode()
    ).hexdigest()[:16]

    return {
        "id": uid,
        "library": library,
        "version": version,             # e.g. "2.0.0", "1.6", "latest"
        "category": category,           # migration | whatsnew | changelog | guide | api
        "source_url": url,
        "chunk_index": chunk_index,
        "total_chunks": total_chunks,
        "text": chunk_text_content,
        "char_count": len(chunk_text_content),
        "collected_at": datetime.now(timezone.utc).isoformat(),
    }


print("Document builder ready.")

Document builder ready.


In [9]:
all_documents: list[dict] = []
failed_urls: list[tuple[str, str]] = []

libs_to_process = [lib for lib in libraries if lib in DOC_URLS]

for lib in tqdm(libs_to_process, desc="Libraries"):
    url_list = DOC_URLS[lib]
    for category, version, url in url_list:
        print(f"  [{lib} v{version}] Fetching ({category}): {url}")
        html = fetch_page(url)
        if html is None:
            failed_urls.append((lib, url))
            continue

        text = extract_main_text(html, url)
        if len(text) < 100:
            print(f"    ⚠ Very short content ({len(text)} chars), skipping.")
            failed_urls.append((lib, url))
            continue

        chunks = chunk_text(text)
        for idx, ch in enumerate(chunks):
            doc = build_document(lib, category, version, url, ch, idx, len(chunks))
            all_documents.append(doc)

        print(f"    ✓ {len(text):,} chars → {len(chunks)} chunks")
        time.sleep(POLITE_DELAY)  # be polite

print(f"\n{'='*60}")
print(f"Total documents (chunks): {len(all_documents)}")
print(f"Failed / skipped URLs:    {len(failed_urls)}")

Libraries:   0%|          | 0/28 [00:00<?, ?it/s]

  [numpy v2.4.0] Fetching (whatsnew): https://numpy.org/doc/stable/release/2.4.0-notes.html


    ✓ 23,635 chars → 22 chunks
  [numpy v2.3.0] Fetching (whatsnew): https://numpy.org/doc/stable/release/2.3.0-notes.html
    ✓ 15,341 chars → 14 chunks
  [numpy v2.0.0] Fetching (migration): https://numpy.org/doc/stable/numpy_2_0_migration_guide.html
    ✓ 18,109 chars → 19 chunks


Libraries:   4%|▎         | 1/28 [00:07<03:10,  7.06s/it]

  [pandas v3.0.0] Fetching (whatsnew): https://pandas.pydata.org/docs/whatsnew/v3.0.0.html
    ✓ 111,393 chars → 110 chunks
  [pandas v2.3.0] Fetching (whatsnew): https://pandas.pydata.org/docs/whatsnew/v2.3.0.html
    ✓ 6,650 chars → 7 chunks


...

  [lightgbm vlatest] Fetching (guide): https://lightgbm.readthedocs.io/en/latest/Python-Intro.html
    ✓ 6,370 chars → 5 chunks


Libraries:  96%|█████████▋| 27/28 [01:35<00:03,  3.05s/it]

  [catboost vlatest] Fetching (guide): https://catboost.ai/en/docs/concepts/python-quickstart
    ✓ 2,450 chars → 2 chunks


Libraries: 100%|██████████| 28/28 [01:38<00:00,  3.53s/it]


Total documents (chunks): 647
Failed / skipped URLs:    4


In [10]:
# ── Summary per library ──────────────────────────────────────────────────────
from collections import Counter

lib_counts = Counter(doc["library"] for doc in all_documents)
cat_counts = Counter(doc["category"] for doc in all_documents)
ver_counts = Counter(f"{doc['library']} v{doc['version']}" for doc in all_documents)

print("Chunks per library:")
for lib, cnt in lib_counts.most_common():
    print(f"  {lib:20s}  {cnt:4d} chunks")

print(f"\nChunks per category:")
for cat, cnt in cat_counts.most_common():
    print(f"  {cat:12s}  {cnt:4d} chunks")

print(f"\nChunks per library+version (top 20):")
for key, cnt in ver_counts.most_common(20):
    print(f"  {key:35s}  {cnt:4d} chunks")

if failed_urls:
    print(f"\nFailed URLs ({len(failed_urls)}):")
    for lib, url in failed_urls:
        print(f"  [{lib}] {url}")

Chunks per library:
  pandas                 117 chunks
  scipy                  116 chunks
  numpy                   55 chunks
  requests                52 chunks
  scikit-learn            51 chunks
  matplotlib              44 chunks
  openpyxl                32 chunks
  torchvision             31 chunks
  pillow                  25 chunks
  langchain-openai        25 chunks
  torch                   21 chunks
  datasets                14 chunks
  seaborn                 11 chunks
  tensorflow              10 chunks
  accelerate               9 chunks
  httpx                    8 chunks
  torchaudio               5 chunks
  xgboost                  5 chunks
  lightgbm                 5 chunks
  scikit-image             4 chunks
  langchain                2 chunks
  catboost                 2 chunks
  transformers             1 chunks
  tqdm                     1 chunks
  opencv-python            1 chunks

Chunks per category:
  whatsnew       354 chunks
  migration      177 chunks
  

## 7 — Preview a Sample Chunk

In [11]:
# Preview the first chunk of each category to verify quality
seen_cats = set()
for doc in all_documents:
    cat = doc["category"]
    if cat not in seen_cats:
        seen_cats.add(cat)
        print(f"\n{'─'*60}")
        print(f"Library : {doc['library']}")
        print(f"Version : {doc['version']}")
        print(f"Category: {cat}")
        print(f"URL     : {doc['source_url']}")
        print(f"Chunk   : {doc['chunk_index']+1}/{doc['total_chunks']}")
        print(f"Chars   : {doc['char_count']}")
        print(f"{'─'*60}")
        # Show first 600 chars
        print(doc["text"][:600])
        if doc["char_count"] > 600:
            print("  [... truncated for preview ...]")


────────────────────────────────────────────────────────────
Library : numpy
Version : 2.4.0
Category: whatsnew
URL     : https://numpy.org/doc/stable/release/2.4.0-notes.html
Chunk   : 1/22
Chars   : 1266
────────────────────────────────────────────────────────────
# NumPy 2.4.0 Release Notes #

The NumPy 2.4.0 release continues the work to improve free threaded Python
support, user dtypes implementation, and annotations. There are many expired
deprecations and bug fixes as well.
This release supports Python versions 3.11-3.14

## Highlights #

Apart from annotations and same_value kwarg, the 2.4 highlights are mostly
of interest to downstream developers. They should help in implementing new user
dtypes.
Many annotation improvements. In particular, runtime signature introspection.
New casting kwarg 'same_value' for casting by value.
New PyUFunc_AddLoopsF
  [... truncated for preview ...]

────────────────────────────────────────────────────────────
Library : numpy
Version : 2.0.0
Cat

## 8 — Save to JSON

The output file is saved in two formats:
1. **Full JSON** — array of document dicts (for programmatic loading)  
2. **JSON-Lines** — one JSON object per line (convenient for streaming into a vector DB)

In [12]:
OUTPUT_DIR = Path("../Datasets")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. Standard JSON array ──────────────────────────────────────────────────
json_path = OUTPUT_DIR / "library_docs_for_vectordb.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(all_documents, f, ensure_ascii=False, indent=2)
print(f"Saved {len(all_documents)} documents → {json_path}  "
      f"({json_path.stat().st_size / 1024:.1f} KB)")

# ── 2. JSON-Lines (one object per line) ─────────────────────────────────────
jsonl_path = OUTPUT_DIR / "library_docs_for_vectordb.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for doc in all_documents:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")
print(f"Saved {len(all_documents)} documents → {jsonl_path}  "
      f"({jsonl_path.stat().st_size / 1024:.1f} KB)")

Saved 647 documents → ..\Datasets\library_docs_for_vectordb.json  (1030.6 KB)
Saved 647 documents → ..\Datasets\library_docs_for_vectordb.jsonl  (994.0 KB)


## 9 — Verify Saved Data

In [13]:
# Reload and verify
with open(json_path, encoding="utf-8") as f:
    loaded = json.load(f)

print(f"Reloaded {len(loaded)} documents from JSON.")
print(f"Keys per document: {list(loaded[0].keys())}")
print(f"\nExample document:")
print(json.dumps(loaded[0], indent=2, ensure_ascii=False)[:1200])

Reloaded 647 documents from JSON.
Keys per document: ['id', 'library', 'version', 'category', 'source_url', 'chunk_index', 'total_chunks', 'text', 'char_count', 'collected_at']

Example document:
{
  "id": "79ea6632131d0f40",
  "library": "numpy",
  "version": "2.4.0",
  "category": "whatsnew",
  "source_url": "https://numpy.org/doc/stable/release/2.4.0-notes.html",
  "chunk_index": 0,
  "total_chunks": 22,
  "text": "# NumPy 2.4.0 Release Notes #\n\nThe NumPy 2.4.0 release continues the work to improve free threaded Python\nsupport, user dtypes implementation, and annotations. There are many expired\ndeprecations and bug fixes as well.\nThis release supports Python versions 3.11-3.14\n\n## Highlights #\n\nApart from annotations and same_value kwarg, the 2.4 highlights are mostly\nof interest to downstream developers. They should help in implementing new user\ndtypes.\nMany annotation improvements. In particular, runtime signature introspection.\nNew casting kwarg 'same_value' for cast

## 10 — Quick Stats & Next Steps

**What was collected:**  
- Migration guides, changelogs, what's-new pages, and quick-start guides for all major dependencies  
- Text was cleaned (HTML stripped, code blocks preserved, headings marked)  
- Content was chunked (~1500 chars with 200-char overlap) for optimal embedding  

**Saved files:**  
- `Datasets/library_docs_for_vectordb.json` — full JSON array  
- `Datasets/library_docs_for_vectordb.jsonl` — JSON-Lines format  

**Each document contains:**  
| Field | Description |
|---|---|
| `id` | Deterministic hash (no duplicates on re-run) |
| `library` | Package name |
| `version` | Release version (e.g. `"2.0.0"`, `"1.6"`) or `"latest"` — enables version-filtered retrieval |
| `category` | `migration` / `whatsnew` / `changelog` / `guide` / `api` |
| `source_url` | Original documentation URL |
| `chunk_index` | Position within the source page |
| `total_chunks` | Total chunks from this page |
| `text` | Cleaned text content |
| `char_count` | Length of text |
| `collected_at` | UTC timestamp |

**Next steps:**  
1. Generate embeddings (e.g. `sentence-transformers` or OpenAI)  
2. Ingest into a vector store (FAISS, ChromaDB, Pinecone, etc.)  
3. Use `library` and `version` as a metadata filter in the retriever for faster, more precise lookups  
4. Use in the RAG pipeline for context-augmented code debugging assistance

In [14]:
# Final stats
total_chars = sum(doc["char_count"] for doc in all_documents)
avg_chunk   = total_chars / len(all_documents) if all_documents else 0

print("="*60)
print("COLLECTION COMPLETE")
print("="*60)
print(f"  Libraries processed : {len(libs_to_process)}")
print(f"  URLs fetched        : {total_urls - len(failed_urls)} / {total_urls}")
print(f"  Total chunks        : {len(all_documents)}")
print(f"  Total characters    : {total_chars:,}")
print(f"  Avg chunk size      : {avg_chunk:,.0f} chars")
print(f"  Output (JSON)       : {json_path}")
print(f"  Output (JSONL)      : {jsonl_path}")
print("="*60)

COLLECTION COMPLETE
  Libraries processed : 28
  URLs fetched        : 35 / 39
  Total chunks        : 647
  Total characters    : 804,087
  Avg chunk size      : 1,243 chars
  Output (JSON)       : ..\Datasets\library_docs_for_vectordb.json
  Output (JSONL)      : ..\Datasets\library_docs_for_vectordb.jsonl
